In [1]:
import pandas as pd
import numpy as np
import cv2

In [2]:
df=pd.read_csv(r"D:\INNOMATICS\CNN\emergency_classification.csv")

In [3]:
df.head()

,image_names,emergency_or_not
0,0.jpg,1
1,1.jpg,1
2,2.jpg,1
3,3.jpg,1
4,4.jpg,1


In [4]:
import os
from tqdm import tqdm

IMG_SIZE = 160

def load_images(df, image_folder):
    images = []
    labels = []

    for i in tqdm(range(len(df))):
        img_name = df.iloc[i]['image_names']
        label = df.iloc[i]['emergency_or_not']

        img_path = os.path.join(image_folder, img_name)

        img = cv2.imread(img_path)

        if img is None:
            print(f"Skipping: {img_path}")
            continue


        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))


        img = img / 255.0

        images.append(img)
        labels.append(label)

    return np.array(images), np.array(labels)

In [5]:
image_folder = r"D:\INNOMATICS\CNN\Dataset\images"

X, y = load_images(df, image_folder)

100%|█████████████████████████████████████████████████████████████████████████████| 2352/2352 [00:05<00:00, 443.23it/s]


In [6]:
X.shape

(2352, 160, 160, 3)

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
from tensorflow.keras.applications import VGG16,MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Flatten,Dropout

In [9]:
base_model=MobileNetV2(weights='imagenet',include_top=False,input_shape=(160,160,3))

In [10]:
base_model.summary()

Model: "mobilenetv2_1.00_160"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 160, 160, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 80, 80, 32)        │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 80, 80, 32)        │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 80, 80, 32)        │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 80, 80, 32)        │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 80, 80, 32)        │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 80, 80, 32)        │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 80, 80, 16)        │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 80, 80, 16)        │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 80, 80, 96)        │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 80, 80, 96)        │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 80, 80, 96)        │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 81, 81, 96)        │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 40, 40, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 2,223,872 (8.48 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [11]:
for layers in base_model.layers:
    layers.trainable=False

In [12]:
base_model.summary()

Model: "mobilenetv2_1.00_160"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 160, 160, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1 (Conv2D)                │ (None, 80, 80, 32)        │             864 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bn_Conv1 (BatchNormalization) │ (None, 80, 80, 32)        │             128 │ Conv1[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ Conv1_relu (ReLU)             │ (None, 80, 80, 32)        │               0 │ bn_Conv1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 80, 80, 32)        │             288 │ Conv1_relu[0][0]           │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_BN    │ (None, 80, 80, 32)        │             128 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_relu  │ (None, 80, 80, 32)        │               0 │ expanded_conv_depthwise_B… │
│ (ReLU)                        │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 80, 80, 16)        │             512 │ expanded_conv_depthwise_r… │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_BN      │ (None, 80, 80, 16)        │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand (Conv2D)       │ (None, 80, 80, 96)        │           1,536 │ expanded_conv_project_BN[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_BN             │ (None, 80, 80, 96)        │             384 │ block_1_expand[0][0]       │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_expand_relu (ReLU)    │ (None, 80, 80, 96)        │               0 │ block_1_expand_BN[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_pad (ZeroPadding2D)   │ (None, 81, 81, 96)        │               0 │ block_1_expand_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ block_1_depthwise             │ (None, 40, 40, 96)        │             864 │ block_1_pad[0][0]          │
│ (DepthwiseConv2D)             │                           │               

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
model=Sequential([base_model,Flatten(),
                 Dense(128,activation='relu'),
                 Dropout(0.5),
                 Dense(1,activation='sigmoid')])

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_160 (Functional)    │ (None, 5, 5, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 32000)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       4,096,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,354,241 (24.24 MB)

 Trainable params: 4,096,257 (15.63 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [16]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32
)

Epoch 1/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 27s 344ms/step - accuracy: 0.8325 - loss: 1.0453 - val_accuracy: 0.9130 - val_loss: 0.1932
Epoch 2/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 20s 335ms/step - accuracy: 0.9160 - loss: 0.2066 - val_accuracy: 0.9321 - val_loss: 0.1903
Epoch 3/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 24s 386ms/step - accuracy: 0.9224 - loss: 0.1902 - val_accuracy: 0.9427 - val_loss: 0.1646
Epoch 4/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 25s 428ms/step - accuracy: 0.9346 - loss: 0.1578 - val_accuracy: 0.9363 - val_loss: 0.1828
Epoch 5/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 22s 366ms/step - accuracy: 0.9495 - loss: 0.1210 - val_accuracy: 0.9363 - val_loss: 0.1993
Epoch 6/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 21s 359ms/step - accuracy: 0.9516 - loss: 0.1134 - val_accuracy: 0.9236 - val_loss: 0.1835
Epoch 7/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 22s 373ms/step - accuracy: 0.9686 - loss: 0.0748 - val_accuracy: 0.9384 - val_loss: 0.1864
Epoch 8/100
59/59 ━━━━━━━━━━━━━━━━━━━━ 22s 365ms/step - accuracy: 0.9676 - loss: 0.0944 - 

In [ ]:
loss, acc = model.evaluate(X_val, y_val)
print("Validation Accuracy:", acc)

In [ ]:
def predict_image(model, img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (160,160))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img)

    if pred > 0.5:
        print("Emergency Vehicle ")
    else:
        print("Non-Emergency Vehicle ")

In [ ]:
image=r"D:\INNOMATICS\CNN\Dataset\images\19.jpg"
predict_image(model,image)

In [ ]:
image=r"D:\INNOMATICS\CNN\Dataset\images\991.jpg"
predict_image(model,image)

In [ ]:
image = r"D:\INNOMATICS\CNN\Dataset\images\1100.jpg"
predict_image(model,image)

## Capturing the pretrained model weights using .keras r .t5 file

In [ ]:
model.save('mobilenet_emergency_classifier.h5')

## Combine the Pre-trained model for classification with YOLO for Object Detection

In [ ]:
from ultralytics import YOLO
from tensorflow.keras.models import load_model


yolo_model = YOLO("yolo26n.pt")


classifier = load_model("mobilenet_emergency_classifier.h5")

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 160

cap = cv2.VideoCapture(r"C:\Users\gsudh\Downloads\1900-151662242.mp4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break


    results = yolo_model(frame)

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            # Crop detected vehicle
            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            crop_resized = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
            crop_resized = crop_resized / 255.0
            crop_resized = np.expand_dims(crop_resized, axis=0)

      
            pred = classifier.predict(crop_resized)[0][0]

            if pred > 0.5:
                label = "Emergency "
                color = (0, 0, 255)
            else:
                label = "Non-Emergency "
                color = (0, 255, 0)


            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                color,
                2
            )

    cv2.imshow("Hybrid Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from tensorflow.keras.models import load_model


yolo_model = YOLO("yolov8n.pt")  
classifier = load_model("mobilenet_emergency_classifier.h5")

IMG_SIZE = 160  

cap = cv2.VideoCapture(0)

frame_count = 0
SKIP_FRAMES = 3  # process every 3rd frame


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1


    if frame_count % SKIP_FRAMES != 0:
        continue


    frame = cv2.resize(frame, (640, 480))


    results = yolo_model(frame, verbose=False)

    crops = []
    boxes_list = []

    for r in results:
        boxes = r.boxes.xyxy.cpu().numpy()

        for box in boxes:
            x1, y1, x2, y2 = map(int, box)

            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                continue

            crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
            crop = crop / 255.0

            crops.append(crop)
            boxes_list.append((x1, y1, x2, y2))


    if len(crops) > 0:
        crops = np.array(crops)
        preds = classifier.predict(crops, verbose=0)

        for i, pred in enumerate(preds):
            x1, y1, x2, y2 = boxes_list[i]

            if pred[0] > 0.5:
                label = "Emergency "
                color = (0, 0, 255)
            else:
                label = "Non-Emergency "
                color = (0, 255, 0)

            # Draw box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Label
            cv2.putText(
                frame,
                label,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                color,
                2
            )

  
    cv2.imshow("Fast Detection", frame)

    if cv2.waitKey(0):
        break

cap.release()
cv2.destroyAllWindows()